In [1]:
import pandas as pd
import numpy as np
df = pd.read_parquet("cfpb_clean.parquet")
print(df.shape)
df.head()

(708603, 7)


,Consumer complaint narrative,label,Product,State,Issue,Sub-issue,Date received
0,Kindly address this issue on my credit report....,credit_issue,Credit reporting or other personal consumer re...,IL,Incorrect information on your report,Information belongs to someone else,01/05/24
1,Urgent : Disputed Inquiries on Credit Report -...,credit_issue,Credit reporting or other personal consumer re...,CA,Improper use of your report,Credit inquiries on your report that you don't...,01/28/24
2,This is XXXX XXXX do not deny my complaint by ...,credit_issue,Credit reporting or other personal consumer re...,PA,Incorrect information on your report,Account status incorrect,12/08/23
3,National credit adjusters after giving then po...,debt_fraud,Debt collection,TX,Attempts to collect debt not owed,Debt was result of identity theft,04/23/19
4,"XXXX 2021, amount of XXXX to collect is Not ac...",scam,Debt collection,KY,False statements or representation,Attempted to collect wrong amount,12/28/23


In [2]:
#Keep only the columns needed for Stage 1 prep
cols_needed = [
    'Consumer complaint narrative',
    'Product',
    'State',
    'Issue',
    'Sub-issue',
    'Date received'
]

df = df[cols_needed].copy()
print(df.shape)

(708603, 6)


In [3]:
#Basic cleaning safeguard
df = df.dropna(subset=['Consumer complaint narrative', 'Issue', 'Product'])
df = df[df['Consumer complaint narrative'].str.strip() != ""]
df = df.drop_duplicates(subset=['Consumer complaint narrative'])

print(df.shape)

(708603, 6)


In [4]:
#Standardize text columns
df['Issue'] = df['Issue'].fillna('').str.strip()
df['Sub-issue'] = df['Sub-issue'].fillna('').str.strip()
df['Product'] = df['Product'].fillna('').str.strip()
df['State'] = df['State'].fillna('UNKNOWN').str.strip()

In [5]:
#Create a combined issue text field for rule-based labeling
df['issue_text'] = (
    df['Issue'].str.lower() + " " + df['Sub-issue'].str.lower()
).str.strip()

df[['Issue', 'Sub-issue', 'issue_text']].head()

,Issue,Sub-issue,issue_text
0,Incorrect information on your report,Information belongs to someone else,incorrect information on your report informati...
1,Improper use of your report,Credit inquiries on your report that you don't...,improper use of your report credit inquiries o...
2,Incorrect information on your report,Account status incorrect,incorrect information on your report account s...
3,Attempts to collect debt not owed,Debt was result of identity theft,attempts to collect debt not owed debt was res...
4,False statements or representation,Attempted to collect wrong amount,false statements or representation attempted t...


In [6]:
#Define fraud-related patterns
fraud_keywords = [
    'fraud',
    'scam',
    'identity theft',
    'stolen identity',
    'unauthorized transaction',
    'unauthorized transactions',
    'unauthorized charge',
    'unauthorized charges',
    'debt was result of identity theft',
    'security freeze',
    'fraud alert',
    'false statements or representation',
    'communication tactics'
]

In [7]:
#Define clearly non-fraud patterns
nonfraud_keywords = [
    'incorrect information on your report',
    'improper use of your report',
    'account status incorrect',
    'managing an account',
    'closing an account',
    'trouble during payment process',
    'problem with a purchase shown on your statement',
    'trouble using your card'
]

In [8]:
#Create fraud and non-fraud flags
fraud_pattern = '|'.join(fraud_keywords)
nonfraud_pattern = '|'.join(nonfraud_keywords)

df['fraud_flag'] = df['issue_text'].str.contains(fraud_pattern, case=False, regex=True)
df['nonfraud_flag'] = df['issue_text'].str.contains(nonfraud_pattern, case=False, regex=True)

print(df[['fraud_flag', 'nonfraud_flag']].mean())

fraud_flag       0.149363
nonfraud_flag    0.746594
dtype: float64


In [9]:
#Assign Stage 1 labels carefully
df['stage1_label'] = np.where(
    (df['fraud_flag'] == True) & (df['nonfraud_flag'] == False), 1,
    np.where(
        (df['fraud_flag'] == False) & (df['nonfraud_flag'] == True), 0,
        np.nan
    )
)

print(df['stage1_label'].value_counts(dropna=False))

stage1_label
0.0    529039
1.0    105839
NaN     73725
Name: count, dtype: int64


In [10]:
#Keep only clearly labeled rows
stage1_df = df.dropna(subset=['stage1_label']).copy()
stage1_df['stage1_label'] = stage1_df['stage1_label'].astype(int)

print(stage1_df.shape)
print(stage1_df['stage1_label'].value_counts())

(634878, 10)
stage1_label
0    529039
1    105839
Name: count, dtype: int64


In [11]:
#Rename target to something clearer
stage1_df = stage1_df.rename(columns={'stage1_label': 'is_fraud'})
print(stage1_df['is_fraud'].value_counts())

is_fraud
0    529039
1    105839
Name: count, dtype: int64


In [12]:
#Check class balance
print(stage1_df['is_fraud'].value_counts())
print(stage1_df['is_fraud'].value_counts(normalize=True).round(4))

is_fraud
0    529039
1    105839
Name: count, dtype: int64
is_fraud
0    0.8333
1    0.1667
Name: proportion, dtype: float64


In [13]:
#Inspect examples from both classes
stage1_df[stage1_df['is_fraud'] == 1][
    ['Issue', 'Sub-issue', 'Consumer complaint narrative']
].sample(5, random_state=42)
stage1_df[stage1_df['is_fraud'] == 0][
    ['Issue', 'Sub-issue', 'Consumer complaint narrative']
].sample(5, random_state=42)

,Issue,Sub-issue,Consumer complaint narrative
997741,Incorrect information on your report,Account information incorrect,My closed XXXX XXXX XXXX credit card ( last X...
799241,Improper use of your report,Credit inquiries on your report that you don't...,"Hi, this is really concerning, there's an unfa..."
787008,Incorrect information on your report,Information belongs to someone else,I am reaching out with absolute frustration re...
148131,Improper use of your report,Credit inquiries on your report that you don't...,Subject : Request for Removal of Unrecognized ...
449245,Improper use of your report,Reporting company used your report improperly,I am writing to formally dispute the accounts ...


In [14]:
#Create modeling dataset without leakage columns
stage1_model_df = stage1_df[
    ['Consumer complaint narrative', 'Product', 'State', 'Date received', 'is_fraud']
].copy()

print(stage1_model_df.shape)
stage1_model_df.head()

(634878, 5)


,Consumer complaint narrative,Product,State,Date received,is_fraud
0,Kindly address this issue on my credit report....,Credit reporting or other personal consumer re...,IL,01/05/24,0
1,Urgent : Disputed Inquiries on Credit Report -...,Credit reporting or other personal consumer re...,CA,01/28/24,0
2,This is XXXX XXXX do not deny my complaint by ...,Credit reporting or other personal consumer re...,PA,12/08/23,0
3,National credit adjusters after giving then po...,Debt collection,TX,04/23/19,1
4,"XXXX 2021, amount of XXXX to collect is Not ac...",Debt collection,KY,12/28/23,1


In [15]:
#Add simple text length feature
stage1_model_df['text_length'] = stage1_model_df['Consumer complaint narrative'].str.len()
stage1_model_df['text_length'].describe()

count    634878.000000
mean       1267.754712
std        1574.348459
min          21.000000
25%         426.000000
50%         878.000000
75%        1597.000000
max       35722.000000
Name: text_length, dtype: float64

In [16]:
#Parse dates
stage1_model_df['Date received'] = pd.to_datetime(
    stage1_model_df['Date received'],
    format='%m/%d/%y',
    errors='coerce'
)

stage1_model_df['year'] = stage1_model_df['Date received'].dt.year
stage1_model_df['month'] = stage1_model_df['Date received'].dt.month

In [17]:
#Final missing-value cleanup
stage1_model_df = stage1_model_df.dropna(subset=['Date received'])
print(stage1_model_df.isna().sum())
print(stage1_model_df.shape)

Consumer complaint narrative    0
Product                         0
State                           0
Date received                   0
is_fraud                        0
text_length                     0
year                            0
month                           0
dtype: int64
(634878, 8)


In [18]:
#Final target check
print(stage1_model_df['is_fraud'].value_counts())
print(stage1_model_df['is_fraud'].value_counts(normalize=True).round(4))

is_fraud
0    529039
1    105839
Name: count, dtype: int64
is_fraud
0    0.8333
1    0.1667
Name: proportion, dtype: float64


In [19]:
#Save Stage 1 dataset
stage1_model_df.to_parquet("cfpb_stage1.parquet", index=False)
print("Saved: cfpb_stage1.parquet")

Saved: cfpb_stage1.parquet
